

# Notebook Overview
Ths notebook demonstrates the second deployment for the KNN Book Reccomendation, my last implmentation failed because of few user dimensions which led to very small distances.
## Objective
In this implementation I aim to increase user dimensions by setting the threshhold for rating to about (20-30) which is more realistic, instead of 200 which drastically reduced the dataset

# Conclusion
This implementaion failed due to several reasons, the main one being mixing up variable names and also using sequential filtering instead of filtering simultenously leading to bad test results.

In [1]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt


# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'


# import csv data into dataframes
df_books = pd.read_csv(books_filename,
    encoding = "ISO-8859-1", # latin encoder
    sep = ";",
    header = 0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'], # specifies columns to use
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'}) # specifies datatype of columns


df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})





--2026-05-20 17:08:32--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.2.33, 104.26.3.33, 172.67.70.149, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.2.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip’

book-crossings.zip  100%[===================>]  24.88M   140MB/s    in 0.2s    

2026-05-20 17:08:33 (140 MB/s) - ‘book-crossings.zip’ saved [26085508/26085508]

Archive:  book-crossings.zip
  inflating: BX-Book-Ratings.csv     
  inflating: BX-Books.csv            
  inflating: BX-Users.csv            


# Original Dataset

In [2]:
df = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    on_bad_lines='skip'
)

/tmp/ipykernel_19524/755777257.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


In [3]:
df.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


In [4]:
df_books.head()

,isbn,title,author
0,0195153448,Classical Mythology,Mark P. O. Morford
1,0002005018,Clara Callan,Richard Bruce Wright
2,0060973129,Decision in Normandy,Carlo D'Este
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata
4,0393045218,The Mummies of Urumchi,E. J. W. Barber


In [5]:
df_books.tail()

,isbn,title,author
271374,0440400988,There's a Bat in Bunk Five,Paula Danziger
271375,0525447644,From One to One Hundred,Teri Sloat
271376,006008667X,Lily Dale : The True Story of the Town that Ta...,Christine Wicker
271377,0192126040,Republic (World's Classics),Plato
271378,0767409752,A Guided Tour of Rene Descartes' Meditations o...,Christopher Biffle


In [6]:
# title to isbn matcher

titles = df_books['title'].tolist()
isbn = df_books['isbn'].tolist()

isbn_title = {code:title for code,title in zip(isbn,titles)}


In [7]:
len(df_books)

271379

In [8]:
len(df_books['isbn'].unique()) #all books are unique

271379

In [9]:
len(isbn)

271379

# Original Ratings Dataset

In [10]:
df_r = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";")
df_r.columns.tolist()

['User-ID', 'ISBN', 'Book-Rating']

In [11]:
len(df_r['User-ID'].unique())

105283

In [12]:
df_ratings.head(n = 20)

,user,isbn,rating
0,276725,034545104X,0.0
1,276726,0155061224,5.0
2,276727,0446520802,0.0
3,276729,052165615X,3.0
4,276729,0521795028,6.0
5,276733,2080674722,0.0
6,276736,3257224281,8.0
7,276737,0600570967,6.0
8,276744,038550120X,7.0
9,276745,342310538,10.0


In [13]:
# Counts occurrences of every user
# same users make ratings on different books
users_counts = df_ratings["user"].value_counts()
print(users_counts[:])

user
11676     13602
198711     7550
153662     6109
98391      5891
35859      5850
          ...  
119573        1
276706        1
276697        1
276679        1
276676        1
Name: count, Length: 105283, dtype: int64


In [14]:
df_ratings.tail(n=3)

,user,isbn,rating
1149777,276709,0515107662,10.0
1149778,276721,0590442449,10.0
1149779,276723,05162443314,8.0


In [15]:
len(df_ratings)

1149780

In [16]:
len(df_ratings['isbn'].unique())

340556

# Addressing Inconsistencies

The Ratings have more unique codes than observed in the original book filenames this means  ```340556 - 271379 = 69177``` will have missing titles

### Checking for Subset Relationship

In [17]:
# Convert unique ISBNs to sets
isbns_in_df_books = set(df_books['isbn'])
isbns_in_df_ratings = set(df_ratings['isbn'])

# Checks if all ISBNs in df_books are present in df_ratings
is_books_subset_of_ratings = isbns_in_df_books.issubset(isbns_in_df_ratings)

print(f"Are all ISBNs from df_books present in df_ratings? {is_books_subset_of_ratings}")


is_ratings_subset_of_books = isbns_in_df_ratings.issubset(isbns_in_df_books)
print(f"Are all ISBNs from df_ratings present in df_books? {is_ratings_subset_of_books}")

if not is_books_subset_of_ratings:
    missing_in_ratings = isbns_in_df_books - isbns_in_df_ratings
    print(f"\nNumber of ISBNs in df_books NOT found in df_ratings: {len(missing_in_ratings)}")

if not is_ratings_subset_of_books:
    missing_in_books = isbns_in_df_ratings - isbns_in_df_books
    print(f"Number of ISBNs in df_ratings NOT found in df_books: {len(missing_in_books)}")

Are all ISBNs from df_books present in df_ratings? False
Are all ISBNs from df_ratings present in df_books? False

Number of ISBNs in df_books NOT found in df_ratings: 1209
Number of ISBNs in df_ratings NOT found in df_books: 70386


# Inference


```Number of ISBNs in df_books NOT found in df_ratings: 1209``` - This shows that they are books that exist by different authors but lack ratings. Hence can't compare to anything for getting right reccommendation

```Number of ISBNs in df_ratings NOT found in df_books: 70386``` - This shows that they are 70386 books that have been rated by users but dont have no record in books meaning status remains unknown which is not desired when making reccomendations. This can be users who made entries either misspelled the isbn code or didnt leave details about those specific books i.e books can't be classified if they miss metadata.

In [18]:
# unique users
len(df_ratings['user'].unique())

105283

In [19]:
# users

user_ids = sorted(df_ratings['user'].unique().tolist())
print(user_ids[:30])
len(user_ids)
print(user_ids[-1])


[2, 7, 8, 9, 10, 12, 14, 16, 17, 19, 20, 22, 23, 26, 32, 36, 38, 39, 42, 44, 51, 53, 56, 64, 67, 68, 69, 70, 73, 75]
278854


In [20]:
len(user_ids)

105283

In [21]:
u = 278854
user_indices = df_ratings[df_ratings['user'] == u].index
print(f"Row numbers (indices) where user {u} occurs: {user_indices.tolist()}")


Row numbers (indices) where user 278854 occurs: [9553, 9554, 9555, 9556, 9557, 9558, 9559, 9560]


In [22]:
# book ids as recorded in the df_book

isbn.sort()
print(isbn[:20])

['0000913154', '0001010565', '0001046438', '0001046713', '000104687X', '0001046934', '0001047213', '0001047647', '0001047663', '0001047868', '0001047973', '000104799X', '0001048082', '0001048473', '0001049879', '0001052039', '0001053736', '0001053744', '0001055607', '0001056107']


# pivotting Dataframe

This approach tilts my dataframe in such a way the column isbn in df_ratings gets unique isbn codes and columns get unique user with the ratings




In [23]:
len(df_ratings)

1149780

In [24]:
book_counts = df_ratings['isbn'].value_counts()
valid_books = book_counts[book_counts >= 100].index
len(valid_books)

731

In [25]:
len(valid_books)

731

In [27]:
df_ratings.shape

(1149780, 3)

In [28]:
# merge titles
# filter book codes that were in df ratings but didnt exist in df books

merged_df = pd.merge(df_ratings, df_books, on='isbn', how='inner')

# drop authors column
merged_df.drop(columns = 'author', axis = 1)

#reorganize dataframe columns

reorganized_df= merged_df[['user','isbn','title','rating']]




In [29]:
print(reorganized_df.shape)

# rows dropped

df_ratings.shape[0] - reorganized_df.shape[0]

(1031175, 4)


118605

In [30]:
# NAN Rows
print(f"NaNs per column before: \n{reorganized_df.isna().sum()}")



NaNs per column before: 
user      0
isbn      0
title     0
rating    0
dtype: int64


In [31]:
reorganized_df.head()

,user,isbn,title,rating
0,276725,034545104X,Flesh Tones: A Novel,0.0
1,276726,0155061224,Rites of Passage,5.0
2,276727,0446520802,The Notebook,0.0
3,276729,052165615X,Help!: Level 1,3.0
4,276729,0521795028,The Amsterdam Connection : Level 4 (Cambridge ...,6.0


In [32]:
# finding threshold for books

ratings_count = reorganized_df['isbn'].value_counts().values

# finding percentile

threshhold = np.percentile(ratings_count,99)

In [33]:
# filtering columns
# only valid isbn codes from the df_books should be included to avoid lack of missing book information
# only books with more than 100 ratings meaning it appears more than 100 times
# only users with 200+ ratings

# dropping users

user_counts = reorganized_df['user'].value_counts()
valid_users = user_counts[user_counts >= 200].index
print(len(valid_users))
filtered_df = reorganized_df[reorganized_df['user'].isin(valid_users)]


# dropping users and books
book_counts = filtered_df['isbn'].value_counts() #returns frequencies of unique books
valid_books = book_counts[book_counts >= 100].index
filtered_df = filtered_df[filtered_df['isbn'].isin(valid_books)]

#filtered_df = filtered_df[filtered_df['isbn'].isin(valid_books)]
filtered_df = filtered_df[filtered_df['user'].isin(valid_users)]
filtered_df.shape
# conversion to categorical columns

filtered_df['isbn'] = filtered_df['isbn'].astype('category')
filtered_df['user'] = filtered_df['user'].astype('category')

# creating sparse representations, avoids memory usage
#mask_zero = pd.SparseDtype(np.float64, 0)
#filtered_df['rating'] = filtered_df['rating'].astype(mask_zero)
# pivotting transforms chosen unique column values to rows, columns and rating

pivot_df = filtered_df.pivot_table(index ='isbn',columns = 'user', values = 'rating',fill_value = 0,observed=True)
sparse_matrix = csr_matrix(pivot_df.values)
'''
brute algorithm for comparing a query to each unique point
'''

from sklearn.neighbors import NearestNeighbors
knn_model = NearestNeighbors(metric='cosine', algorithm='brute')
knn_model.fit(sparse_matrix)

# getting relevant titles

# filtering dataframe with only relevant book codes and titles

df = df_books[df_books['isbn'].isin(pivot_df.index)]
df.shape
# getting relevant titles

# filtering dataframe with only relevant book codes and titles

df = df_books[df_books['isbn'].isin(pivot_df.index)]
df.shape

# title to isbn dictionary

titles = df['title'].tolist()
isbn = df['isbn'].tolist()

isbn_title = {code:title for code,title in zip(isbn,titles)}

title_isbn = {title:code for code,title in isbn_title.items()}

#get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")



816


In [34]:
from sklearn import neighbors

## function to return recommended books - this will be tested
def get_recommends(book):

 #check if title is valid
  if book not in title_isbn:
    print("Enter valid title")
    return

  # create the first dimension
  recommended_books = [book]

  # second dimension

  neighbors = []

  # get book_code

  book_code = title_isbn[book]

  # get row index for book code

  row_index = pivot_df.index.get_loc(book_code)

  # get csr vector i.e where the book is located

  position_vector = sparse_matrix[row_index]

  # find neighbors

  distances, indices = knn_model.kneighbors(position_vector, n_neighbors= 10)


  # flatten distances and indices from 2D to 1D

  distances, indices = distances.flatten(), indices.flatten()


  for index,row in enumerate(indices):

   # skips the first iteration to avoid recording the query point as a neighbor
    if index == 0:
      continue

    # get book codes of neighbours

    n_code = pivot_df.index[row]

    # neighbors titles

    n_title = isbn_title[n_code]

    # append to second dimension along with their distances as a 3rd dimension(list)

    neighbors.append([n_title, float(distances[index])])


  # append to main title

  recommended_books.append(neighbors)

  return recommended_books

In [36]:
# I Know This Much Is True elememnt
print(len(sparse_matrix[27].data))

18


In [39]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()

["Where the Heart Is (Oprah's Book Club (Paperback))", [['The Lovely Bones: A Novel', 0.715856671333313], ['Bel Canto: A Novel', 0.8146533370018005], ['The Joy Luck Club', 0.8157694339752197], ["The Pilot's Wife : A Novel", 0.8168827891349792], ['The Notebook', 0.8192572593688965], ["The Book of Ruth (Oprah's Book Club (Paperback))", 0.822733998298645], ['The Reader', 0.824946939945221], ['The Red Tent (Bestselling Backlist)', 0.8305545449256897], ["She's Come Undone (Oprah's Book Club (Paperback))", 0.8335007429122925]]]
You haven't passed yet. Keep trying!
